In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.6f}".format)

PROJECT_DIR = Path.cwd().parent
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

asset_prices_usd = pd.read_csv(
    RAW_DIR / "asset_prices_usd.csv",
    index_col=0,
    parse_dates=True
)

gbpusd = pd.read_csv(
    RAW_DIR / "gbpusd_exchange_rate.csv",
    index_col=0,
    parse_dates=True
).squeeze("columns")

gbpusd.name = "GBPUSD"

print("USD asset prices:", asset_prices_usd.shape)
print("GBP/USD exchange rate:", gbpusd.shape)

USD asset prices: (1254, 5)
GBP/USD exchange rate: (1299,)


In [4]:
# Match FX observations to the US asset trading calendar
gbpusd_aligned = gbpusd.reindex(asset_prices_usd.index)

# Check whether any asset trading dates lack an FX observation
missing_fx_dates = gbpusd_aligned[gbpusd_aligned.isna()]

print("Asset trading dates without FX observations:", len(missing_fx_dates))

if len(missing_fx_dates) > 0:
    display(missing_fx_dates)
    raise ValueError("Some asset trading dates have no matching FX observation.")

# Confirm date alignment
assert asset_prices_usd.index.equals(gbpusd_aligned.index)

print("Date alignment successful.")
print("Aligned observations:", len(gbpusd_aligned))

Asset trading dates without FX observations: 1


Date
2025-04-21   NaN
Name: GBPUSD, dtype: float64

ValueError: Some asset trading dates have no matching FX observation.

In [5]:
# Identify the dates present in the asset dataset
# but absent from the FX dataset

missing_fx_dates = asset_prices_usd.index.difference(gbpusd.index)

print("Number of missing FX dates:", len(missing_fx_dates))

print("\nDates without matching FX observations:")
display(missing_fx_dates)

# Inspect the observations immediately before and after
# the first missing date

if len(missing_fx_dates) > 0:

    first_missing_date = missing_fx_dates[0]

    print("\nFirst missing date:", first_missing_date)

    print("\nAsset prices around the missing date:")
    display(
        asset_prices_usd.loc[
            first_missing_date - pd.Timedelta(days=3):
            first_missing_date + pd.Timedelta(days=3)
        ]
    )

    print("\nAvailable FX observations around the missing date:")
    display(
        gbpusd.loc[
            first_missing_date - pd.Timedelta(days=3):
            first_missing_date + pd.Timedelta(days=3)
        ]
    )

Number of missing FX dates: 1

Dates without matching FX observations:


DatetimeIndex(['2025-04-21'], dtype='datetime64[us]', name='Date', freq=None)


First missing date: 2025-04-21 00:00:00

Asset prices around the missing date:


,GLD,MSFT,QQQ,SPY,TLT
Date,,,,,
2025-04-21,315.589996,354.953156,430.102325,505.495087,80.383438
2025-04-22,311.109985,362.563812,441.393372,518.646912,80.832077
2025-04-23,303.649994,370.045990,451.403381,526.683716,81.617256
2025-04-24,308.070007,382.806183,464.104584,537.769836,82.477165



Available FX observations around the missing date:


Date
2025-04-22   1.337972
2025-04-23   1.327316
2025-04-24   1.326453
Name: GBPUSD, dtype: float64

In [2]:
### Inspect missing FX observations date
missing_date = pd.Timestamp("2025-04-21")

previous_fx_date = gbpusd.index[gbpusd.index < missing_date].max()
next_fx_date = gbpusd.index[gbpusd.index > missing_date].min()

print("Missing date:", missing_date)

print("\nPrevious available FX observation:")
print(previous_fx_date, gbpusd.loc[previous_fx_date])

print("\nNext available FX observation:")
print(next_fx_date, gbpusd.loc[next_fx_date])

print("\nCalendar-day gap from previous observation:")
print((missing_date - previous_fx_date).days)

NameError: name 'pd' is not defined

In [ ]:
# Reindex FX data to the US asset trading calendar
gbpusd_aligned = gbpusd.reindex(asset_prices_usd.index)

# Count missing observations before filling
missing_before = gbpusd_aligned.isna().sum()

# Carry forward the immediately preceding available observation
gbpusd_aligned = gbpusd_aligned.ffill(limit=1)

# Count missing observations after filling
missing_after = gbpusd_aligned.isna().sum()

print("Missing FX observations before filling:", missing_before)
print("Missing FX observations after filling:", missing_after)

# Confirm successful alignment
assert gbpusd_aligned.index.equals(asset_prices_usd.index)
assert missing_after == 0

print("\nFX and asset price dates successfully aligned.")
print("Aligned observations:", len(gbpusd_aligned))

print("\nExchange rate used on 21 April 2025:")
print(gbpusd_aligned.loc["2025-04-21"])